# Combined 55x55 BP/RP Truncation Visuals

This notebook renders the combined independent BP/RP truncation figures using Plotly.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

try:
    from IPython.display import Javascript, HTML, display
except Exception:
    Javascript = None
    HTML = None
    display = None



def find_project_root(start=None):
    """Locate the curated prep repository root from a notebook or script working directory."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "prep" / "55x55_experiments").exists():
            return candidate / "prep"
        if (candidate / "55x55_experiments").exists() and (candidate / "k55_experiments").exists():
            return candidate
    raise FileNotFoundError("Could not locate the prep repository root.")


PROJECT_ROOT = find_project_root()
GRID_DIR = PROJECT_ROOT / "55x55_experiments"
K55_DIR = PROJECT_ROOT / "k55_experiments"
FIGURE_DIR = GRID_DIR / "combined_55x55_visuals_out" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print("PROJECT_ROOT:", PROJECT_ROOT)


# Force inline rendering (avoid browser pop-out)
preferred_renderers = ["vscode", "notebook_connected", "plotly_mimetype", "jupyterlab"]
available_renderers = set(pio.renderers)
for r in preferred_renderers:
    if r in available_renderers:
        pio.renderers.default = r
        break

# Shared high-quality modebar export settings (applies to integrated camera/download button)
EXPORT_BASE_WIDTH = 1600
EXPORT_BASE_HEIGHT = 900
EXPORT_SCALE = 2.4  # 1600x900 * 2.4 = 3840x2160


def _slugify_filename(text):
    s = "".join(ch.lower() if str(ch).isalnum() else "_" for ch in str(text))
    s = s.strip("_")
    return s or "plot"


def _hires_export_config(filename, width=EXPORT_BASE_WIDTH, height=EXPORT_BASE_HEIGHT, scale=EXPORT_SCALE, image_format="png"):
    return {
        "displaylogo": False,
        "toImageButtonOptions": {
            "format": str(image_format),
            "filename": _slugify_filename(filename),
            "width": int(width),
            "height": int(height),
            "scale": float(scale),
        },
    }


def _show_fig_hires(fig, filename="plot", width=None, height=None, scale=EXPORT_SCALE, image_format="png", auto_save=False, save_dir=FIGURE_DIR):
    w = int(width or fig.layout.width or EXPORT_BASE_WIDTH)
    h = int(height or fig.layout.height or EXPORT_BASE_HEIGHT)
    if auto_save:
        out_dir = Path(save_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f"{_slugify_filename(filename)}.{str(image_format).lower()}"
        try:
            fig.write_image(str(out_path), format=str(image_format), width=w, height=h, scale=scale)
            print(f"Saved figure to: {out_path.resolve()}")
        except Exception as e:
            print(f"Auto-save failed ({e}). Install kaleido to enable static export.")
    fig.show(config=_hires_export_config(filename=filename, width=w, height=h, scale=scale, image_format=image_format))

def _show_fig_hires_with_live_camera(fig, filename="plot", width=None, height=None, scale=EXPORT_SCALE, scene_id="scene", image_format="png"):
    """Render figure with a built-in live camera monitor panel bound to this exact plot."""
    w = int(width or fig.layout.width or EXPORT_BASE_WIDTH)
    h = int(height or fig.layout.height or EXPORT_BASE_HEIGHT)
    cfg = _hires_export_config(filename=filename, width=w, height=h, scale=scale, image_format=image_format)

    # Fallback if HTML display is unavailable.
    if HTML is None or display is None:
        fig.show(config=cfg)
        return

    scene_literal = json.dumps(str(scene_id))
    post_script = f"""
(function() {{
  var gd = document.getElementById('{{plot_id}}');
  if (!gd) return;

  var panelId = 'live-camera-panel-{{plot_id}}';
  var old = document.getElementById(panelId);
  if (old) old.remove();

  var panel = document.createElement('div');
  panel.id = panelId;
  panel.style.cssText = [
    'position:fixed',
    'right:14px',
    'bottom:14px',
    'z-index:9999',
    'font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace',
    'font-size:12px',
    'line-height:1.25',
    'color:#0b2a4a',
    'background:rgba(255,255,255,0.96)',
    'border:1px solid rgba(90,110,130,0.35)',
    'border-radius:8px',
    'box-shadow:0 4px 18px rgba(10,20,30,0.15)',
    'padding:8px',
    'min-width:360px',
    'max-width:460px'
  ].join(';');

  panel.innerHTML = `
    <div style="display:flex;align-items:center;justify-content:space-between;margin-bottom:6px;gap:8px;">
      <strong>Live Camera</strong>
      <div style="display:flex;gap:6px;">
        <button id="cam-copy-{{plot_id}}" style="padding:2px 8px;border:1px solid #9db3c7;background:#f7fbff;border-radius:6px;cursor:pointer;">Copy</button>
        <button id="cam-close-{{plot_id}}" style="padding:2px 8px;border:1px solid #9db3c7;background:#f7fbff;border-radius:6px;cursor:pointer;">Close</button>
      </div>
    </div>
    <pre id="cam-pre-{{plot_id}}" style="margin:0;max-height:280px;overflow:auto;white-space:pre-wrap;">Loading...</pre>
  `;
  document.body.appendChild(panel);

  var pre = panel.querySelector('#cam-pre-{{plot_id}}');
  var copyBtn = panel.querySelector('#cam-copy-{{plot_id}}');
  var closeBtn = panel.querySelector('#cam-close-{{plot_id}}');
  var sceneId = {scene_literal};

  function getCamera() {{
    var sceneLayout = ((gd.layout || {{}})[sceneId]) || ((gd._fullLayout || {{}})[sceneId]) || null;
    var cam = sceneLayout && sceneLayout.camera ? sceneLayout.camera : null;
    if (!cam && sceneLayout && sceneLayout._scene && typeof sceneLayout._scene.getCamera === 'function') {{
      cam = sceneLayout._scene.getCamera();
    }}
    return cam || null;
  }}

  function render() {{
    var cam = getCamera();
    if (!cam) {{
      pre.textContent = 'Camera unavailable for this figure.';
      panel.dataset.assign = '';
      return;
    }}
    var assign = 'CUSTOM_3D_CAMERA = ' + JSON.stringify(cam);
    panel.dataset.assign = assign;
    pre.textContent = assign + '

' + JSON.stringify(cam, null, 2);
  }}

  var interval = setInterval(render, 140);
  if (typeof gd.on === 'function') {{
    gd.on('plotly_relayout', render);
    gd.on('plotly_afterplot', render);
  }}
  render();

  copyBtn.addEventListener('click', function() {{
    var txt = panel.dataset.assign || '';
    if (!txt) return;
    console.log(txt);
    if (navigator.clipboard && navigator.clipboard.writeText) {{
      navigator.clipboard.writeText(txt).catch(function(){{}});
    }}
    copyBtn.textContent = 'Copied';
    setTimeout(function() {{ copyBtn.textContent = 'Copy'; }}, 900);
  }});

  closeBtn.addEventListener('click', function() {{
    clearInterval(interval);
    try {{
      if (typeof gd.removeListener === 'function') gd.removeListener('plotly_relayout', render);
      if (typeof gd.off === 'function') gd.off('plotly_relayout', render);
    }} catch (e) {{}}
    panel.remove();
  }});
}})();
"""

    html = fig.to_html(
        full_html=False,
        include_plotlyjs="cdn",
        config=cfg,
        post_script=post_script,
        default_width=f"{w}px",
        default_height=f"{h}px",
    )
    display(HTML(html))


def capture_last_plotly_camera(scene_id="scene", copy_to_clipboard=True, wait_ms=8000):
    """Capture camera from the latest rendered Plotly figure after manual rotation.

    Usage:
        1) Rotate/zoom the 3D figure by hand.
        2) Run: capture_last_plotly_camera()
        3) Paste printed line into: CUSTOM_3D_CAMERA = {...}
    """
    if Javascript is None or display is None:
        raise RuntimeError("IPython display/Javascript is unavailable in this environment")

    scene_literal = json.dumps(str(scene_id))
    copy_flag = "true" if copy_to_clipboard else "false"
    wait_ms = int(wait_ms)
    js = f"""
    (async () => {{
      const sceneId = {scene_literal};
      const maxMs = {wait_ms};

      const getPlots = () => {{
        const sels = '.js-plotly-plot, .plotly-graph-div';
        const out = [];
        const pushFromDoc = (doc) => {{
          if (!doc) return;
          out.push(...Array.from(doc.querySelectorAll(sels)));
        }};
        pushFromDoc(document);
        try {{
          if (window.parent && window.parent !== window) pushFromDoc(window.parent.document);
        }} catch (e) {{}}
        return out;
      }};

      const sleep = (ms) => new Promise(r => setTimeout(r, ms));
      let plots = getPlots();
      const t0 = Date.now();
      while (!plots.length && Date.now() - t0 < maxMs) {{
        await sleep(120);
        plots = getPlots();
      }}
      if (!plots.length) {{
        alert('No Plotly figure found after waiting. Re-run the 3D cell, then try again.');
        return;
      }}

      const gd = plots[plots.length - 1];
      const sceneLayout = ((gd.layout || {{}})[sceneId]) || ((gd._fullLayout || {{}})[sceneId]) || null;
      let cam = sceneLayout && sceneLayout.camera ? sceneLayout.camera : null;
      if (!cam && sceneLayout && sceneLayout._scene && typeof sceneLayout._scene.getCamera === 'function') {{
        cam = sceneLayout._scene.getCamera();
      }}
      if (!cam) {{
        alert(`Could not read camera for ${{sceneId}} on the latest figure.`);
        return;
      }}
      const txt = JSON.stringify(cam);
      const assign = 'CUSTOM_3D_CAMERA = ' + txt;
      console.log(assign);
      if ({copy_flag} && navigator.clipboard && navigator.clipboard.writeText) {{
        navigator.clipboard.writeText(assign).catch(() => {{}});
      }}
      alert('Camera captured. A ready-to-paste line was logged to browser console' + ({copy_flag} ? ' and copied to clipboard.' : '.'));
    }})();
    """
    display(Javascript(js))


def watch_last_plotly_camera_live(scene_id="scene", fixed_panel=True, wait_ms=10000):
    """Show live camera values while rotating the latest Plotly 3D figure.

    Usage:
        1) Render a 3D Plotly figure.
        2) This watcher may be auto-opened by the 3D cell, or run manually.
        3) Rotate/zoom; values update live.
        4) Click copy button and paste into CUSTOM_3D_CAMERA.
    """
    if Javascript is None or display is None:
        raise RuntimeError("IPython display/Javascript is unavailable in this environment")

    scene_literal = json.dumps(str(scene_id))
    fixed_flag = "true" if fixed_panel else "false"
    wait_ms = int(wait_ms)

    js = f"""
    (async () => {{
      const sceneId = {scene_literal};
      const fixedPanel = {fixed_flag};
      const maxMs = {wait_ms};

      const getPlots = () => {{
        const sels = '.js-plotly-plot, .plotly-graph-div';
        const out = [];
        const pushFromDoc = (doc) => {{
          if (!doc) return;
          out.push(...Array.from(doc.querySelectorAll(sels)));
        }};
        pushFromDoc(document);
        try {{
          if (window.parent && window.parent !== window) pushFromDoc(window.parent.document);
        }} catch (e) {{}}
        return out;
      }};

      const sleep = (ms) => new Promise(r => setTimeout(r, ms));
      let plots = getPlots();
      const t0 = Date.now();
      while (!plots.length && Date.now() - t0 < maxMs) {{
        await sleep(120);
        plots = getPlots();
      }}
      if (!plots.length) {{
        alert('No Plotly figure found after waiting. Re-run the 3D cell and keep output visible.');
        return;
      }}

      const gd = plots[plots.length - 1];

      const uid = Math.random().toString(36).slice(2, 9);
      const panel = document.createElement('div');
      panel.id = `camera-monitor-${{uid}}`;

      const baseStyle = [
        'z-index:9999',
        'font-family:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace',
        'font-size:12px',
        'line-height:1.25',
        'color:#0b2a4a',
        'background:rgba(255,255,255,0.96)',
        'border:1px solid rgba(90,110,130,0.35)',
        'border-radius:8px',
        'box-shadow:0 4px 18px rgba(10,20,30,0.15)',
        'padding:8px',
        'min-width:320px',
        'max-width:420px'
      ];
      if (fixedPanel) {{
        baseStyle.push('position:fixed', 'right:14px', 'bottom:14px');
      }} else {{
        baseStyle.push('position:relative', 'margin:8px 0');
      }}
      panel.style.cssText = baseStyle.join(';');

      panel.innerHTML = `
        <div style="display:flex;align-items:center;justify-content:space-between;margin-bottom:6px;gap:8px;">
          <strong>Live Camera (${{sceneId}})</strong>
          <div style="display:flex;gap:6px;">
            <button id="cam-copy-${{uid}}" style="padding:2px 8px;border:1px solid #9db3c7;background:#f7fbff;border-radius:6px;cursor:pointer;">Copy</button>
            <button id="cam-close-${{uid}}" style="padding:2px 8px;border:1px solid #9db3c7;background:#f7fbff;border-radius:6px;cursor:pointer;">Close</button>
          </div>
        </div>
        <pre id="cam-pre-${{uid}}" style="margin:0;max-height:260px;overflow:auto;white-space:pre-wrap;">Loading...</pre>
      `;

      if (fixedPanel) {{
        document.body.appendChild(panel);
      }} else {{
        const host = gd.parentElement || gd;
        host.insertAdjacentElement('beforebegin', panel);
      }}

      const pre = panel.querySelector(`#cam-pre-${{uid}}`);
      const copyBtn = panel.querySelector(`#cam-copy-${{uid}}`);
      const closeBtn = panel.querySelector(`#cam-close-${{uid}}`);

      const getCamera = () => {{
        const sceneLayout = ((gd.layout || {{}})[sceneId]) || ((gd._fullLayout || {{}})[sceneId]) || null;
        let cam = sceneLayout && sceneLayout.camera ? sceneLayout.camera : null;
        if (!cam && sceneLayout && sceneLayout._scene && typeof sceneLayout._scene.getCamera === 'function') {{
          cam = sceneLayout._scene.getCamera();
        }}
        return cam || null;
      }};

      const toAssign = (cam) => `CUSTOM_3D_CAMERA = ${{JSON.stringify(cam)}}`;

      const render = () => {{
        const cam = getCamera();
        if (!cam) {{
          pre.textContent = 'Camera unavailable for this figure.';
          panel.dataset.assign = '';
          return;
        }}
        pre.textContent = JSON.stringify(cam, null, 2);
        panel.dataset.assign = toAssign(cam);
      }};

      const onRelayout = () => render();
      if (typeof gd.on === 'function') gd.on('plotly_relayout', onRelayout);
      render();

      copyBtn.addEventListener('click', () => {{
        const txt = panel.dataset.assign || '';
        if (!txt) return;
        console.log(txt);
        if (navigator.clipboard && navigator.clipboard.writeText) navigator.clipboard.writeText(txt).catch(() => {{}});
        copyBtn.textContent = 'Copied';
        setTimeout(() => copyBtn.textContent = 'Copy', 900);
      }});

      closeBtn.addEventListener('click', () => {{
        try {{
          if (typeof gd.removeListener === 'function') gd.removeListener('plotly_relayout', onRelayout);
          if (typeof gd.off === 'function') gd.off('plotly_relayout', onRelayout);
        }} catch (e) {{}}
        panel.remove();
      }});
    }})();
    """
    display(Javascript(js))




# Shared settings
STAT = "mean"
CRITERION = "youden"
REVERSE_Y = False
DIAG_X = [1, 55]
DIAG_Y = [1, 55]

# Shared 3D view for surface+contour cells (reuse for LR/RF so angle is identical).
SHARED_3D_CAMERA = dict(
    center=dict(x=0, y=0, z=0),
    eye=dict(x=1.38, y=1.03, z=0.78),
    up=dict(x=0, y=0, z=1),
)
SHARED_3D_ASPECT = dict(x=1.22, y=1.16, z=0.66)

# Optional override: set this to your preferred camera dict once, and all 3D cells use it.
# Example:
# CUSTOM_3D_CAMERA = dict(center=dict(x=0, y=0, z=0), eye=dict(x=1.45, y=0.95, z=0.72), up=dict(x=0, y=0, z=1))
CUSTOM_3D_CAMERA = None

METHOD_ORDER = ["LR", "KNN", "SVM", "RF"]

METHODS = {
    "SVM": {
        "kind": "single",
        "path": GRID_DIR / "svm_55x55_truncation_out" / "truncation_svm_bp_rp_grid_raw.parquet",
        "title": "SVM contour map",
        "exclude": lambda g: ~((g["K_bp"].astype(int) == 1) & (g["K_rp"].astype(int) == 1)),
    },
    "LR": {
        "kind": "single",
        "path": GRID_DIR / "lr_55x55_truncation_out" / "truncation_logreg_bp_rp_grid_raw.parquet",
        "title": "Logistic Regression contour map",
        "exclude": lambda g: ~((g["K_bp"].astype(int) == 1) & (g["K_rp"].astype(int) == 1)),
    },
    "RF": {
        "kind": "single",
        "path": GRID_DIR / "rf_55x55_truncation_out" / "truncation_rf_bp_rp_grid_raw.parquet",
        "title": "Random Forest contour map",
        "exclude": lambda g: ~((g["K_rp"].astype(int) == 1) & (g["K_bp"].astype(int).isin([1, 2, 3, 4, 5]))),
    },
    "KNN": {
        "kind": "single",
        "path": GRID_DIR / "knn_55x55_truncation_out" / "truncation_knn_bp_rp_grid_raw.parquet",
        "title": "KNN contour map",
        "exclude": lambda g: ~((g["K_bp"].astype(int) == 1) & (g["K_rp"].astype(int) == 1)),
    },
}

COLLAGE_SPECS = [
    {"name": "PR AUC", "metric": "test_PR_AUC", "criterion": CRITERION, "reverse_scale": False},
    {"name": "Brier Score", "metric": "test_Brier", "criterion": CRITERION, "reverse_scale": True},
    {"name": "Accuracy", "metric": "accuracy", "criterion": CRITERION, "reverse_scale": False},
    {"name": "F1 Score", "metric": "f1_score", "criterion": CRITERION, "reverse_scale": False},
    {"name": "Youden Score", "metric": "youden_score", "criterion": CRITERION, "reverse_scale": False},
]


In [ ]:
def q05(x):
    return float(np.nanquantile(x, 0.05))


def q50(x):
    return float(np.nanquantile(x, 0.50))


def q95(x):
    return float(np.nanquantile(x, 0.95))


def _load_bp_rp_grid(spec):
    kind = spec.get("kind", "single")
    if kind == "single":
        return pd.read_parquet(spec["path"])
    raise ValueError(f"Unsupported source kind: {kind}")


def _normalize_metric_name(metric_name):
    return str(metric_name).strip().lower()


def _needs_derived_metric(metric_name):
    m = _normalize_metric_name(metric_name)
    return m in {
        "accuracy", "acc", "test_accuracy", "test_acc",
        "f1", "f1_score", "test_f1", "test_f1_score",
        "youden", "youden_score", "test_youden", "test_youden_score",
    }


def _with_derived_metric(df, metric_name, criterion="youden"):
    if metric_name in df.columns:
        return df

    if not _needs_derived_metric(metric_name):
        return df

    pref = f"{criterion}_test_"
    c_tp, c_tn, c_fp, c_fn = [f"{pref}{s}" for s in ("tp", "tn", "fp", "fn")]
    needed = [c_tp, c_tn, c_fp, c_fn]

    if not all(c in df.columns for c in needed):
        miss = [c for c in needed if c not in df.columns]
        raise KeyError(f"missing columns for derived metric ({criterion}): {miss}")

    out = df.copy()

    tp = out[c_tp]
    tn = out[c_tn]
    fp = out[c_fp]
    fn = out[c_fn]

    m = _normalize_metric_name(metric_name)

    if m in {"accuracy", "acc", "test_accuracy", "test_acc"}:
        denom = tp + tn + fp + fn
        out[metric_name] = np.where(denom > 0, (tp + tn) / denom, np.nan)
    elif m in {"f1", "f1_score", "test_f1", "test_f1_score"}:
        denom = 2.0 * tp + fp + fn
        out[metric_name] = np.where(denom > 0, (2.0 * tp) / denom, np.nan)
    elif m in {"youden", "youden_score", "test_youden", "test_youden_score"}:
        tpr_denom = tp + fn
        tnr_denom = tn + fp
        tpr = np.where(tpr_denom > 0, tp / tpr_denom, np.nan)
        tnr = np.where(tnr_denom > 0, tn / tnr_denom, np.nan)
        out[metric_name] = tpr + tnr - 1.0

    return out


def _prepare_bp_rp_grid(df, method_name, metric="test_PR_AUC", criterion="youden", stat="mean", reverse_y=False):
    df = _with_derived_metric(df, metric_name=metric, criterion=criterion)

    c_bp, c_rp = "K_bp", "K_rp"
    cols = [c_bp, c_rp, metric]
    if not all(c in df.columns for c in cols):
        miss = [c for c in cols if c not in df.columns]
        raise KeyError(f"{method_name}: missing columns {miss}")

    d = df[cols].copy()
    d[c_bp] = d[c_bp].astype(int)
    d[c_rp] = d[c_rp].astype(int)

    g = (
        d.groupby([c_bp, c_rp], as_index=False)[metric]
         .agg(q05=q05, q50=q50, q95=q95, mean="mean", std="std")
    )

    g = g[METHODS[method_name]["exclude"](g)].copy()

    zcol = stat if stat in g.columns else "mean"
    title_metric = metric.replace("test_", "")

    piv = g.pivot(index=c_rp, columns=c_bp, values=zcol)
    piv05 = g.pivot(index=c_rp, columns=c_bp, values="q05").reindex(index=piv.index, columns=piv.columns)
    piv50 = g.pivot(index=c_rp, columns=c_bp, values="q50").reindex(index=piv.index, columns=piv.columns)
    piv95 = g.pivot(index=c_rp, columns=c_bp, values="q95").reindex(index=piv.index, columns=piv.columns)

    if reverse_y:
        piv = piv.sort_index(ascending=False)
        piv05 = piv05.reindex(index=piv.index)
        piv50 = piv50.reindex(index=piv.index)
        piv95 = piv95.reindex(index=piv.index)

    X = piv.columns.to_numpy(int)
    Y = piv.index.to_numpy(int)
    Z = piv.to_numpy(float)

    q05_arr = piv05.to_numpy(float)
    q50_arr = piv50.to_numpy(float)
    q95_arr = piv95.to_numpy(float)
    band = q95_arr - q05_arr
    custom = np.stack([q05_arr, q50_arr, q95_arr, band], axis=-1)

    return {
        "X": X,
        "Y": Y,
        "Z": Z,
        "custom": custom,
        "title_metric": title_metric,
        "zcol": zcol,
        "g": g,
    }


def _axis_edges(vals):
    vals = np.asarray(vals, dtype=float)
    if vals.size == 0:
        return [0.0, 1.0]
    if vals.size == 1:
        return [vals[0] - 0.5, vals[0] + 0.5]
    step = np.median(np.diff(np.sort(vals)))
    step = float(step) if np.isfinite(step) and step > 0 else 1.0
    return [float(vals.min() - step / 2.0), float(vals.max() + step / 2.0)]


def _finite_vals(z):
    zz = z[np.isfinite(z)]
    return zz if zz.size else np.array([0.0, 1.0], dtype=float)


def _range_from_vals(vals, mode="robust_5_95"):
    v = _finite_vals(np.asarray(vals, dtype=float))
    if mode == "robust_2_98":
        lo, hi = np.quantile(v, [0.02, 0.98])
    elif mode == "robust_10_90":
        lo, hi = np.quantile(v, [0.10, 0.90])
    elif mode == "robust_5_95":
        lo, hi = np.quantile(v, [0.05, 0.95])
    else:
        lo, hi = np.min(v), np.max(v)

    lo, hi = float(lo), float(hi)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        m = float(np.nanmean(v))
        return (m - 1e-9, m + 1e-9)
    return lo, hi


def _metric_lower_is_better(metric_name):
    s = str(metric_name).lower()
    return ("brier" in s) or ("logloss" in s) or ("loss" in s)


def _topk(g, zcol, title_metric, k=3):
    flat = g[["K_bp", "K_rp", zcol]].dropna().copy()
    lower_is_better = _metric_lower_is_better(title_metric)
    top = flat.sort_values(zcol, ascending=lower_is_better).head(int(k)).copy()
    top["rank"] = np.arange(1, len(top) + 1)
    return top, lower_is_better


In [ ]:
# Load each method once
raw_by_method = {}
for name in METHOD_ORDER:
    df = _load_bp_rp_grid(METHODS[name])
    raw_by_method[name] = df
    print(f"{name}: loaded {len(df):,} rows")


In [ ]:
# Design cell: default style, collages (PR AUC + Brier + Accuracy + F1 + Youden)
required = [
    "raw_by_method", "METHOD_ORDER", "METHODS", "COLLAGE_SPECS", "REVERSE_Y",
    "DIAG_X", "DIAG_Y", "_prepare_bp_rp_grid", "_axis_edges", "_topk",
    "_finite_vals", "_range_from_vals", "go", "make_subplots", "np", "pd",
]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError("Run setup cells first. Missing: " + ", ".join(missing))

BASE_STYLE = {
    "template": "plotly_white",
    "colorscale": "YlGnBu",
    "range_mode": "robust_5_95",
    "contour_color": "rgba(255,255,255,0.80)",
    "grid_color": "rgba(30,30,30,0.08)",
}

# Poster-oriented palette options (Vilnius University magenta + cosmic blues).
POSTER_COLOR_THEMES = {
    "vilnius_rose": [
        [0.00, "#f9f4f7"], [0.20, "#f0dbe6"], [0.42, "#e0a8c1"],
        [0.64, "#be5b89"], [0.84, "#8b1e58"], [1.00, "#5a0036"],
    ],
    "cosmic_blue": [
        [0.00, "#f1f7fa"], [0.20, "#d3eaf4"], [0.42, "#98cde2"],
        [0.64, "#4f9ec5"], [0.84, "#1f5b87"], [1.00, "#0b2f54"],
    ],
    "rose_to_navy": [
        [0.00, "#f8f2f6"], [0.20, "#ecd6e3"], [0.40, "#d59fbe"],
        [0.62, "#9d7dac"], [0.82, "#486892"], [1.00, "#153a62"],
    ],
    "paper_blush": [
        [0.00, "#fffaf8"], [0.22, "#f6e8eb"], [0.44, "#e9c8d4"],
        [0.66, "#cf8daa"], [0.84, "#a44672"], [1.00, "#73003f"],
    ],
}

DEFAULT_COLOR_THEME = "vilnius_rose"
print("Available color themes:", ", ".join(POSTER_COLOR_THEMES.keys()))


LABEL_BOX_VARIANTS = {
    "soft_chip": {
        "font_size": 17,
        "font_family": "Arial, Helvetica, sans-serif",
        "font_color": "#0b3c5d",
        "bg": "rgba(255,255,255,0.90)",
        "border": "rgba(90,110,130,0.22)",
        "border_width": 1,
        "pad": 6,
    },
    "navy_pill": {
        "font_size": 16,
        "font_family": "Arial, Helvetica, sans-serif",
        "font_color": "#ffffff",
        "bg": "rgba(11,60,93,0.86)",
        "border": "rgba(255,255,255,0.48)",
        "border_width": 1,
        "pad": 7,
    },
    "minimal_outline": {
        "font_size": 17,
        "font_family": "Arial, Helvetica, sans-serif",
        "font_color": "#0b3c5d",
        "bg": "rgba(255,255,255,0.55)",
        "border": "rgba(11,60,93,0.50)",
        "border_width": 1,
        "pad": 6,
    },
}


def _add_dark_badges(fig, row, col, top_df, value_col):
    if len(top_df) == 0:
        return

    df = top_df[["K_bp", "K_rp", "rank", value_col]].copy()
    df["K_bp"] = df["K_bp"].astype(float)
    df["K_rp"] = df["K_rp"].astype(float)
    df["rank"] = df["rank"].astype(int)
    df["_dx"] = 0.0
    df["_dy"] = 0.0

    # If multiple ranks land on the same point, spread badges slightly
    # so numbers stay readable while still indicating the same hotspot.
    spread = 0.42
    grouped = df.groupby(["K_bp", "K_rp"], sort=False).groups
    for _, idx in grouped.items():
        ids = list(idx)
        if len(ids) <= 1:
            continue

        # Assign offsets in rank order (1 is best).
        ids = sorted(ids, key=lambda i: int(df.at[i, "rank"]))
        n = len(ids)
        if n == 2:
            offsets = [(-spread, 0.0), (spread, 0.0)]
        elif n == 3:
            offsets = [(-spread, spread), (spread, spread), (0.0, -spread)]
        else:
            angles = np.linspace(0.0, 2.0 * np.pi, n, endpoint=False)
            offsets = [(spread * np.cos(a), spread * np.sin(a)) for a in angles]

        for i, (dx, dy) in zip(ids, offsets):
            df.at[i, "_dx"] = float(dx)
            df.at[i, "_dy"] = float(dy)

    # Draw subtle leader lines from true point to offset badge positions.
    moved = df[(df["_dx"].abs() > 1e-12) | (df["_dy"].abs() > 1e-12)]
    for _, rec in moved.iterrows():
        x0 = float(rec["K_bp"])
        y0 = float(rec["K_rp"])
        x1 = x0 + float(rec["_dx"])
        y1 = y0 + float(rec["_dy"])
        fig.add_trace(
            go.Scatter(
                x=[x0, x1],
                y=[y0, y1],
                mode="lines",
                line=dict(color="rgba(255,255,255,0.62)", width=1.0),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=row,
            col=col,
        )

    # Draw lower priority first so rank 1 remains on top.
    for _, rec in df.sort_values("rank", ascending=False).iterrows():
        rank = int(rec["rank"])
        x_true = int(round(float(rec["K_bp"])))
        y_true = int(round(float(rec["K_rp"])))
        x = float(rec["K_bp"] + rec["_dx"])
        y = float(rec["K_rp"] + rec["_dy"])
        val = float(rec[value_col])

        msize = 17 if rank == 1 else (15 if rank == 2 else 14)

        fig.add_trace(
            go.Scatter(
                x=[x],
                y=[y],
                mode="markers+text",
                text=[str(rank)],
                textposition="middle center",
                textfont=dict(color="white", size=9),
                marker=dict(
                    size=msize,
                    symbol="circle",
                    color="#0b3c5d",
                    line=dict(color="white", width=1.6),
                    opacity=0.96,
                ),
                customdata=[[x_true, y_true, val]],
                hovertemplate=(
                    "Rank=%{text}<br>"
                    "K_bp=%{customdata[0]}<br>"
                    "K_rp=%{customdata[1]}<br>"
                    "value=%{customdata[2]:.4f}<extra></extra>"
                ),
                showlegend=False,
            ),
            row=row,
            col=col,
        )



def _render_collage(spec, label_box="soft_chip", title_override=None, color_theme=DEFAULT_COLOR_THEME):
    metric = spec["metric"]
    criterion = spec["criterion"]
    metric_name = spec["name"]
    reverse_scale = bool(spec.get("reverse_scale", False))
    lb = LABEL_BOX_VARIANTS.get(label_box, LABEL_BOX_VARIANTS["soft_chip"])
    colorscale = POSTER_COLOR_THEMES.get(color_theme, BASE_STYLE["colorscale"])

    panels = {}
    for model_name in METHOD_ORDER:
        panels[model_name] = _prepare_bp_rp_grid(
            raw_by_method[model_name],
            method_name=model_name,
            metric=metric,
            criterion=criterion,
            stat="mean",
            reverse_y=REVERSE_Y,
        )

    hs = 0.008
    vs = 0.012
    fig = make_subplots(
        rows=2,
        cols=2,
        horizontal_spacing=hs,
        vertical_spacing=vs,
    )

    fig_h = 620
    margin_l, margin_r, margin_t, margin_b = 6, 26, 6, 6
    inner_h = fig_h - margin_t - margin_b

    y_span = (1.0 - vs) / 2.0
    col_gap = 0.10
    outer_pad = 0.018
    sq_w = (1.0 - 2.0 * outer_pad - col_gap) / 2.0
    inner_w = y_span * inner_h / sq_w
    fig_w = int(round(inner_w + margin_l + margin_r))

    col1_dom = (outer_pad, outer_pad + sq_w)
    col2_dom = (col1_dom[1] + col_gap, col1_dom[1] + col_gap + sq_w)
    row1_dom = (y_span + vs, 1.0)
    row2_dom = (0.0, y_span)

    for idx, model_name in enumerate(METHOD_ORDER, start=1):
        row = 1 if idx <= 2 else 2
        col = 1 if idx in (1, 3) else 2

        d = panels[model_name]
        X, Y, Z = d["X"], d["Y"], d["Z"]
        custom = d["custom"]
        title_metric, zcol = d["title_metric"], d["zcol"]

        if np.isnan(Z).any():
            Z = (
                pd.DataFrame(Z)
                .interpolate(axis=0, limit_direction="both")
                .interpolate(axis=1, limit_direction="both")
                .to_numpy(float)
            )
            if np.isnan(Z).any():
                Z = np.nan_to_num(Z, nan=float(np.nanmean(Z)))

        lo, hi = _range_from_vals(_finite_vals(Z), mode=BASE_STYLE["range_mode"])

        x_dom = col1_dom if col == 1 else col2_dom
        y_dom = row1_dom if row == 1 else row2_dom
        x_domain_end = x_dom[1]
        cbar_x = x_domain_end + 0.008
        cbar_y = (y_dom[0] + y_dom[1]) / 2.0
        cbar_len = (y_dom[1] - y_dom[0]) * 0.84
        cbar_ticks = np.linspace(float(lo), float(hi), 5).tolist()
        cbar_ticktext = [f"{v:.3f}" for v in cbar_ticks]

        fig.add_trace(
            go.Heatmap(
                x=X,
                y=Y,
                z=Z,
                colorscale=colorscale,
                reversescale=reverse_scale,
                zsmooth="best",
                connectgaps=True,
                xgap=0,
                ygap=0,
                customdata=custom,
                zmin=lo,
                zmax=hi,
                colorbar=dict(
                    len=cbar_len,
                    y=cbar_y,
                    x=cbar_x,
                    yanchor="middle",
                    xanchor="left",
                    thickness=7,
                    outlinewidth=0,
                    tickmode="array",
                    tickvals=cbar_ticks,
                    ticktext=cbar_ticktext,
                    tickfont=dict(size=13),
                ),
                hovertemplate=(
                    "K_bp=%{x}<br>K_rp=%{y}<br>"
                    f"{title_metric}_{zcol}=%{{z:.4f}}<extra></extra>"
                ),
            ),
            row=row,
            col=col,
        )

        fig.add_trace(
            go.Contour(
                x=X,
                y=Y,
                z=Z,
                contours=dict(coloring="none", showlabels=True),
                line=dict(color=BASE_STYLE["contour_color"], width=1),
                connectgaps=True,
                showscale=False,
                hoverinfo="skip",
            ),
            row=row,
            col=col,
        )

        top_df, _ = _topk(d["g"], zcol, title_metric, k=3)
        _add_dark_badges(fig, row, col, top_df, zcol)

        # Faint dual diagonal: still visible on dark and light regions.
        fig.add_trace(
            go.Scatter(
                x=DIAG_X,
                y=DIAG_Y,
                mode="lines",
                line=dict(color="rgba(255,255,255,0.34)", width=1.6),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=row,
            col=col,
        )
        fig.add_trace(
            go.Scatter(
                x=DIAG_X,
                y=DIAG_Y,
                mode="lines",
                line=dict(color="rgba(25,25,25,0.32)", width=0.8),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=row,
            col=col,
        )

        xlo, xhi = _axis_edges(X)
        ylo, yhi = _axis_edges(Y)
        xref = "x" if idx == 1 else f"x{idx}"
        yref = "y" if idx == 1 else f"y{idx}"

        panel_title = {"LR": "Logistic Regression", "KNN": "KNN", "SVM": "SVM", "RF": "Random Forest"}.get(model_name, model_name)
        fig.add_annotation(
            x=xlo + 1.0,
            y=yhi - 0.9,
            xref=xref,
            yref=yref,
            text=panel_title,
            showarrow=False,
            xanchor="left",
            yanchor="top",
            font=dict(size=lb["font_size"], color=lb["font_color"], family=lb.get("font_family", "Arial, Helvetica, sans-serif")),
            bgcolor=lb["bg"],
            bordercolor=lb["border"],
            borderwidth=lb["border_width"],
            borderpad=lb["pad"],
        )

        fig.update_xaxes(
            title_text="K<sub>BP</sub>" if row == 2 else None,
            range=[xlo, xhi],
            domain=list(x_dom),
            constrain="domain",
            showgrid=True,
            showticklabels=(row == 2),
            title_font=dict(size=19),
            tickfont=dict(size=14),
            gridcolor=BASE_STYLE["grid_color"],
            zeroline=False,
            row=row,
            col=col,
        )
        fig.update_yaxes(
            title_text="K<sub>RP</sub>" if col == 1 else None,
            range=[ylo, yhi],
            domain=list(y_dom),
            scaleanchor=xref,
            scaleratio=1,
            constrain="domain",
            showgrid=True,
            showticklabels=(col == 1),
            title_font=dict(size=19),
            tickfont=dict(size=14),
            gridcolor=BASE_STYLE["grid_color"],
            zeroline=False,
            row=row,
            col=col,
        )

    fig.update_layout(
        template=BASE_STYLE["template"],
        paper_bgcolor="white",
        plot_bgcolor="white",
        title=None,
        width=fig_w,
        height=fig_h,
        margin=dict(l=margin_l, r=margin_r, t=margin_t, b=margin_b),
        font=dict(family="Avenir Next, Inter, Segoe UI, sans-serif", size=11),
        showlegend=False,
    )

    _show_fig_hires(fig, filename=f"{metric_name}_contour_maps", width=fig_w, height=fig_h)


for spec in COLLAGE_SPECS:
    _render_collage(spec, color_theme=DEFAULT_COLOR_THEME)


In [ ]:
# Separate cell: collages for Logistic Regression + Random Forest only
required = [
    "raw_by_method", "METHODS", "COLLAGE_SPECS", "REVERSE_Y",
    "DIAG_X", "DIAG_Y", "_prepare_bp_rp_grid", "_axis_edges", "_topk",
    "_finite_vals", "_range_from_vals", "_add_dark_badges", "go", "make_subplots", "np", "pd",
    "POSTER_COLOR_THEMES", "BASE_STYLE", "LABEL_BOX_VARIANTS", "DEFAULT_COLOR_THEME", "_show_fig_hires",
]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError("Run setup/design cells first. Missing: " + ", ".join(missing))

METHOD_ORDER_LR_RF = ["LR", "RF"]

def _render_collage_lr_rf(spec, label_box="soft_chip", color_theme=DEFAULT_COLOR_THEME):
    metric = spec["metric"]
    criterion = spec["criterion"]
    metric_name = spec["name"]
    reverse_scale = bool(spec.get("reverse_scale", False))
    lb = LABEL_BOX_VARIANTS.get(label_box, LABEL_BOX_VARIANTS["soft_chip"])
    colorscale = POSTER_COLOR_THEMES.get(color_theme, BASE_STYLE["colorscale"])

    panels = {}
    for model_name in METHOD_ORDER_LR_RF:
        panels[model_name] = _prepare_bp_rp_grid(
            raw_by_method[model_name],
            method_name=model_name,
            metric=metric,
            criterion=criterion,
            stat="mean",
            reverse_y=REVERSE_Y,
        )

    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.06)

    fig_h = 460
    margin_l, margin_r, margin_t, margin_b = 6, 26, 6, 6
    inner_h = fig_h - margin_t - margin_b

    col_gap = 0.08
    outer_pad = 0.025
    sq_w = (1.0 - 2.0 * outer_pad - col_gap) / 2.0
    inner_w = inner_h / sq_w
    fig_w = int(round(inner_w + margin_l + margin_r))

    col1_dom = (outer_pad, outer_pad + sq_w)
    col2_dom = (col1_dom[1] + col_gap, col1_dom[1] + col_gap + sq_w)
    y_dom = (0.0, 1.0)

    for idx, model_name in enumerate(METHOD_ORDER_LR_RF, start=1):
        col = idx
        d = panels[model_name]
        X, Y, Z = d["X"], d["Y"], d["Z"]
        custom = d["custom"]
        title_metric, zcol = d["title_metric"], d["zcol"]

        if np.isnan(Z).any():
            Z = (
                pd.DataFrame(Z)
                .interpolate(axis=0, limit_direction="both")
                .interpolate(axis=1, limit_direction="both")
                .to_numpy(float)
            )
            if np.isnan(Z).any():
                Z = np.nan_to_num(Z, nan=float(np.nanmean(Z)))

        lo, hi = _range_from_vals(_finite_vals(Z), mode=BASE_STYLE["range_mode"])

        x_dom = col1_dom if col == 1 else col2_dom
        cbar_x = x_dom[1] + 0.008
        cbar_ticks = np.linspace(float(lo), float(hi), 5).tolist()
        cbar_ticktext = [f"{v:.3f}" for v in cbar_ticks]

        fig.add_trace(
            go.Heatmap(
                x=X,
                y=Y,
                z=Z,
                colorscale=colorscale,
                reversescale=reverse_scale,
                zsmooth="best",
                connectgaps=True,
                xgap=0,
                ygap=0,
                customdata=custom,
                zmin=lo,
                zmax=hi,
                colorbar=dict(
                    len=0.84,
                    y=0.50,
                    x=cbar_x,
                    yanchor="middle",
                    xanchor="left",
                    thickness=8,
                    outlinewidth=0,
                    tickmode="array",
                    tickvals=cbar_ticks,
                    ticktext=cbar_ticktext,
                    tickfont=dict(size=13),
                ),
                hovertemplate=(
                    "K_bp=%{x}<br>K_rp=%{y}<br>"
                    f"{title_metric}_{zcol}=%{{z:.4f}}<extra></extra>"
                ),
            ),
            row=1,
            col=col,
        )

        fig.add_trace(
            go.Contour(
                x=X,
                y=Y,
                z=Z,
                contours=dict(coloring="none", showlabels=True),
                line=dict(color=BASE_STYLE["contour_color"], width=1),
                connectgaps=True,
                showscale=False,
                hoverinfo="skip",
            ),
            row=1,
            col=col,
        )

        top_df, _ = _topk(d["g"], zcol, title_metric, k=3)
        _add_dark_badges(fig, 1, col, top_df, zcol)

        fig.add_trace(
            go.Scatter(
                x=DIAG_X,
                y=DIAG_Y,
                mode="lines",
                line=dict(color="rgba(255,255,255,0.34)", width=1.6),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=1,
            col=col,
        )
        fig.add_trace(
            go.Scatter(
                x=DIAG_X,
                y=DIAG_Y,
                mode="lines",
                line=dict(color="rgba(25,25,25,0.32)", width=0.8),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=1,
            col=col,
        )

        xlo, xhi = _axis_edges(X)
        ylo, yhi = _axis_edges(Y)
        xref = "x" if idx == 1 else f"x{idx}"
        yref = "y" if idx == 1 else f"y{idx}"

        panel_title = {"LR": "Logistic Regression", "RF": "Random Forest"}.get(model_name, model_name)
        fig.add_annotation(
            x=xlo + 1.0,
            y=yhi - 0.9,
            xref=xref,
            yref=yref,
            text=panel_title,
            showarrow=False,
            xanchor="left",
            yanchor="top",
            font=dict(size=lb["font_size"], color=lb["font_color"], family=lb.get("font_family", "Arial, Helvetica, sans-serif")),
            bgcolor=lb["bg"],
            bordercolor=lb["border"],
            borderwidth=lb["border_width"],
            borderpad=lb["pad"],
        )

        fig.update_xaxes(
            title_text="K<sub>BP</sub>",
            range=[xlo, xhi],
            domain=list(x_dom),
            constrain="domain",
            showgrid=True,
            showticklabels=True,
            title_font=dict(size=19),
            tickfont=dict(size=14),
            gridcolor=BASE_STYLE["grid_color"],
            zeroline=False,
            row=1,
            col=col,
        )
        fig.update_yaxes(
            title_text="K<sub>RP</sub>" if col == 1 else None,
            range=[ylo, yhi],
            domain=list(y_dom),
            scaleanchor=xref,
            scaleratio=1,
            constrain="domain",
            showgrid=True,
            showticklabels=(col == 1),
            title_font=dict(size=19),
            tickfont=dict(size=14),
            gridcolor=BASE_STYLE["grid_color"],
            zeroline=False,
            row=1,
            col=col,
        )

    fig.update_layout(
        template=BASE_STYLE["template"],
        paper_bgcolor="white",
        plot_bgcolor="white",
        title=None,
        width=fig_w,
        height=fig_h,
        margin=dict(l=margin_l, r=margin_r, t=margin_t, b=margin_b),
        font=dict(family="Avenir Next, Inter, Segoe UI, sans-serif", size=11),
        showlegend=False,
    )

    metric_name_norm = "".join(ch.lower() for ch in str(metric_name) if ch.isalnum())
    is_pr_auc = metric_name_norm in {"prauc", "precisionrecallauc"}
    _show_fig_hires(
        fig,
        filename=f"{metric_name}_contour_maps_lr_rf",
        width=fig_w,
        height=fig_h,
        image_format=("svg" if is_pr_auc else "png"),
        auto_save=is_pr_auc,
        save_dir=FIGURE_DIR,
    )


for spec in COLLAGE_SPECS:
    _render_collage_lr_rf(spec, color_theme=DEFAULT_COLOR_THEME)


In [ ]:
# Logistic Regression + Random Forest PR AUC: coupled 3D surface + contour companions
required = [
    "raw_by_method", "_prepare_bp_rp_grid", "go", "make_subplots", "np", "pd", "REVERSE_Y", "CRITERION", "Path", "_axis_edges",
    "SHARED_3D_CAMERA", "SHARED_3D_ASPECT", "CUSTOM_3D_CAMERA", "_show_fig_hires", "Javascript", "display",
]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError("Run setup/load cells first. Missing: " + ", ".join(missing))

metric = "test_PR_AUC"

METHOD_ROWS = [
    {
        "method": "LR",
        "label": "Logistic Regression",
        "slice_path": K55_DIR / "lr_k55_truncation_out" / "truncation_logreg_raw.parquet",
    },
    {
        "method": "RF",
        "label": "Random Forest",
        "slice_path": K55_DIR / "rf_k55_truncation_out" / "truncation_rf_raw.parquet",
    },
]


def _build_surface_panel(method_name, slice_path, metric_name):
    d = _prepare_bp_rp_grid(
        raw_by_method[method_name],
        method_name=method_name,
        metric=metric_name,
        criterion=CRITERION,
        stat="mean",
        reverse_y=REVERSE_Y,
    )

    X, Y, Z = d["X"], d["Y"], d["Z"]
    zcol = d["zcol"]

    if np.isnan(Z).any():
        Z = (
            pd.DataFrame(Z)
            .interpolate(axis=0, limit_direction="both")
            .interpolate(axis=1, limit_direction="both")
            .to_numpy(float)
        )
        if np.isnan(Z).any():
            Z = np.nan_to_num(Z, nan=float(np.nanmean(Z)))

    zvals = Z[np.isfinite(Z)]
    zmin, zmax = (float(np.min(zvals)), float(np.max(zvals))) if zvals.size else (0.0, 1.0)
    pad = max((zmax - zmin) * 0.06, 0.002)
    z_floor = zmin - 2.2 * pad

    slice_k = np.array([], dtype=int)
    slice_z = np.array([], dtype=float)
    if slice_path.exists():
        sraw = pd.read_parquet(slice_path)
        if metric_name in sraw.columns and "K" in sraw.columns:
            s = sraw.groupby("K", as_index=False)[metric_name].mean().sort_values("K")
            s = s[s["K"].isin(X)].copy()
            slice_k = s["K"].to_numpy(int)
            slice_z = s[metric_name].to_numpy(float)

    if slice_k.size == 0:
        gdiag = d["g"].copy()
        gdiag = gdiag[gdiag["K_bp"].astype(int) == gdiag["K_rp"].astype(int)]
        if not gdiag.empty:
            gdiag = gdiag.sort_values("K_bp")
            slice_k = gdiag["K_bp"].astype(int).to_numpy()
            slice_z = gdiag[zcol].to_numpy(float)

    return {
        "X": X,
        "Y": Y,
        "Z": Z,
        "zcol": zcol,
        "g": d["g"],
        "zmin": zmin,
        "zmax": zmax,
        "pad": pad,
        "z_floor": z_floor,
        "slice_k": slice_k,
        "slice_z": slice_z,
    }


panels = [{"meta": m, "data": _build_surface_panel(m["method"], m["slice_path"], metric)} for m in METHOD_ROWS]


def _top3_badges_from_grid(g, zcol):
    flat = g[["K_bp", "K_rp", zcol]].dropna().copy()
    if flat.empty:
        return flat.assign(rank=[])
    top = flat.sort_values(zcol, ascending=False).head(3).copy()
    top["rank"] = np.arange(1, len(top) + 1)
    return top


def _add_rank_badges_2d(fig_obj, row, col, top_df, value_col):
    if top_df is None or len(top_df) == 0:
        return
    for _, rec in top_df.sort_values("rank", ascending=False).iterrows():
        rank = int(rec["rank"])
        msize = 22 if rank == 1 else (20 if rank == 2 else 19)
        fig_obj.add_trace(
            go.Scatter(
                x=[float(rec["K_bp"])],
                y=[float(rec["K_rp"])],
                mode="markers+text",
                text=[str(rank)],
                textposition="middle center",
                textfont=dict(color="white", size=11),
                marker=dict(
                    size=msize,
                    symbol="circle",
                    color="#0b3c5d",
                    line=dict(color="white", width=1.6),
                    opacity=0.96,
                ),
                customdata=[[int(round(float(rec["K_bp"]))), int(round(float(rec["K_rp"]))), float(rec[value_col])]],
                hovertemplate=(
                    "Rank=%{text}<br>"
                    "K_BP=%{customdata[0]}<br>"
                    "K_RP=%{customdata[1]}<br>"
                    "value=%{customdata[2]:.4f}<extra></extra>"
                ),
                showlegend=False,
            ),
            row=row,
            col=col,
        )


def _add_rank_badges_3d(fig_obj, row, col, top_df, value_col, z_bump):
    if top_df is None or len(top_df) == 0:
        return
    for _, rec in top_df.sort_values("rank", ascending=False).iterrows():
        rank = int(rec["rank"])
        msize = 8 if rank == 1 else (7 if rank == 2 else 6)
        z_val = float(rec[value_col]) + float(z_bump)
        fig_obj.add_trace(
            go.Scatter3d(
                x=[float(rec["K_bp"])],
                y=[float(rec["K_rp"])],
                z=[z_val],
                mode="markers+text",
                text=[str(rank)],
                textposition="middle center",
                textfont=dict(color="white", size=11),
                marker=dict(
                    size=msize,
                    color="#0b3c5d",
                    line=dict(color="white", width=4),
                    opacity=0.98,
                ),
                hovertemplate=(
                    "Rank=%{text}<br>"
                    "K_BP=%{x}<br>"
                    "K_RP=%{y}<br>"
                    "value=%{z:.4f}<extra></extra>"
                ),
                showlegend=False,
            ),
            row=row,
            col=col,
        )


fig = make_subplots(
    rows=2,
    cols=2,
    column_widths=[0.52, 0.48],
    vertical_spacing=0.08,
    horizontal_spacing=0.03,
    specs=[[{"type": "xy"}, {"type": "scene"}], [{"type": "xy"}, {"type": "scene"}]],
)

for row_idx, pack in enumerate(panels, start=1):
    meta = pack["meta"]
    d = pack["data"]
    X, Y, Z = d["X"], d["Y"], d["Z"]
    zcol = d["zcol"]
    top_df = _top3_badges_from_grid(d["g"], zcol)
    zmin, zmax, z_floor, pad = d["zmin"], d["zmax"], d["z_floor"], d["pad"]
    slice_k, slice_z = d["slice_k"], d["slice_z"]

    fig.add_trace(
        go.Surface(
            x=X, y=Y, z=Z,
            colorscale="YlGnBu", cmin=zmin, cmax=zmax, opacity=0.98,
            showscale=True,
            colorbar=dict(
                title="PR AUC",
                thickness=14,
                len=0.36,
                y=(0.79 if row_idx == 1 else 0.21),
                x=0.98,
                tickformat=".3f",
                tickfont=dict(size=12),
                outlinewidth=0,
            ),
            lighting=dict(ambient=0.58, diffuse=0.78, specular=0.20, roughness=0.82, fresnel=0.10),
            lightposition=dict(x=130, y=120, z=170),
            contours=dict(z=dict(show=True, usecolormap=False, color="rgba(255,255,255,0.55)", width=1, highlight=False, project=dict(z=True))),
            hovertemplate=f"{meta['label']}<br>K_BP=%{{x}}<br>K_RP=%{{y}}<br>PR_AUC_mean=%{{z:.4f}}<extra></extra>",
            name=f"{meta['label']} surface",
        ),
        row=row_idx, col=2,
    )

    fig.add_trace(
        go.Surface(
            x=X, y=Y, z=np.full_like(Z, z_floor, dtype=float),
            surfacecolor=Z, colorscale="YlGnBu", cmin=zmin, cmax=zmax,
            opacity=0.62, showscale=False, hoverinfo="skip",
            lighting=dict(ambient=0.95, diffuse=0.08, specular=0.0, roughness=1.0, fresnel=0.0),
            name=f"{meta['label']} base",
        ),
        row=row_idx, col=2,
    )

    if slice_k.size:
        fig.add_trace(
            go.Scatter3d(x=slice_k, y=slice_k, z=np.full_like(slice_z, z_floor, dtype=float), mode="lines",
                         line=dict(color="rgba(8,25,42,0.50)", width=4), hoverinfo="skip", showlegend=False),
            row=row_idx, col=2,
        )

        guide_step = max(1, len(slice_k) // 16)
        for k, zv in zip(slice_k[::guide_step], slice_z[::guide_step]):
            fig.add_trace(
                go.Scatter3d(x=[k, k], y=[k, k], z=[z_floor, float(zv)], mode="lines",
                             line=dict(color="rgba(20,45,65,0.36)", width=2), hoverinfo="skip", showlegend=False),
                row=row_idx, col=2,
            )

        fig.add_trace(
            go.Scatter3d(x=slice_k, y=slice_k, z=slice_z, mode="lines",
                         line=dict(color="rgba(255,255,255,0.86)", width=10), hoverinfo="skip", showlegend=False),
            row=row_idx, col=2,
        )
        fig.add_trace(
            go.Scatter3d(x=slice_k, y=slice_k, z=slice_z, mode="lines+markers",
                         line=dict(color="rgba(8,25,42,0.95)", width=5),
                         marker=dict(size=4.0, color="#0b3c5d", line=dict(color="white", width=0.9)),
                         hovertemplate=f"{meta['label']}<br>K=%{{x}}<br>PR_AUC_mean=%{{z:.4f}}<extra></extra>",
                         showlegend=False),
            row=row_idx, col=2,
        )

        fig.add_trace(
        go.Heatmap(
            x=X, y=Y, z=Z,
            colorscale="YlGnBu", zmin=zmin, zmax=zmax, zsmooth="best", connectgaps=True,
            xgap=0, ygap=0, showscale=False,
            hovertemplate=f"{meta['label']}<br>K_BP=%{{x}}<br>K_RP=%{{y}}<br>PR_AUC_mean=%{{z:.4f}}<extra></extra>",
        ),
        row=row_idx, col=1,
    )

    fig.add_trace(
        go.Contour(
            x=X, y=Y, z=Z,
            contours=dict(coloring="none", showlabels=True),
            line=dict(color="rgba(255,255,255,0.80)", width=1),
            connectgaps=True, showscale=False, hoverinfo="skip",
        ),
        row=row_idx, col=1,
    )

    _add_rank_badges_2d(fig, row_idx, 1, top_df, zcol)

    diag_xy = slice_k if slice_k.size else X
    if diag_xy.size:
        fig.add_trace(go.Scatter(x=diag_xy, y=diag_xy, mode="lines", line=dict(color="rgba(255,255,255,0.72)", width=3.2), hoverinfo="skip", showlegend=False), row=row_idx, col=1)
        fig.add_trace(go.Scatter(x=diag_xy, y=diag_xy, mode="lines", line=dict(color="rgba(12,12,12,0.62)", width=1.8), hoverinfo="skip", showlegend=False), row=row_idx, col=1)

    xlo, xhi = _axis_edges(X)
    ylo, yhi = _axis_edges(Y)
    fig.update_xaxes(title_text="K_BP", range=[xlo, xhi], constrain="domain", showgrid=True, gridcolor="rgba(90,110,130,0.16)", zeroline=False, tickfont=dict(size=18), row=row_idx, col=1)
    fig.update_yaxes(title_text="K_RP", range=[ylo, yhi], scaleanchor=("x" if row_idx == 1 else "x2"), scaleratio=1, constrain="domain", showgrid=True, gridcolor="rgba(90,110,130,0.16)", zeroline=False, tickfont=dict(size=18), row=row_idx, col=1)

    fig.update_scenes(
        xaxis=dict(title="K_BP", showbackground=True, backgroundcolor="rgba(246,249,252,1)", gridcolor="rgba(90,110,130,0.18)", zerolinecolor="rgba(90,110,130,0.24)", showspikes=False, tickmode="linear", dtick=6, tickfont=dict(size=12)),
        yaxis=dict(title="K_RP", showbackground=True, backgroundcolor="rgba(246,249,252,1)", gridcolor="rgba(90,110,130,0.18)", zerolinecolor="rgba(90,110,130,0.24)", showspikes=False, tickmode="linear", dtick=6, tickfont=dict(size=12)),
        zaxis=dict(title="PR AUC", range=[z_floor - 0.2 * pad, zmax + pad], showbackground=True, backgroundcolor="rgba(250,252,255,1)", gridcolor="rgba(90,110,130,0.16)", zerolinecolor="rgba(90,110,130,0.20)", showspikes=False, tickformat=".3f", nticks=7, tickfont=dict(size=12)),
        aspectmode="manual", aspectratio=SHARED_3D_ASPECT, camera=(CUSTOM_3D_CAMERA or SHARED_3D_CAMERA), dragmode="turntable",
        row=row_idx, col=2,
    )

fig.update_layout(template="plotly_white", width=1760, height=1900, font=dict(family="Avenir Next, Inter, Segoe UI, sans-serif", size=15), margin=dict(l=18, r=18, t=24, b=18), showlegend=False)

fig.update_layout(
    scene_camera=(CUSTOM_3D_CAMERA or SHARED_3D_CAMERA),
    scene2_camera=(CUSTOM_3D_CAMERA or SHARED_3D_CAMERA),
    scene_aspectmode="manual",
    scene2_aspectmode="manual",
    scene_aspectratio=SHARED_3D_ASPECT,
    scene2_aspectratio=SHARED_3D_ASPECT,
)

_show_fig_hires(fig, filename="lr_rf_pr_auc_surface_contour_linked", width=1760, height=1900, image_format="svg", auto_save=True, save_dir=FIGURE_DIR)

# Link the two 3D scenes (scene <-> scene2) so rotating one rotates the other.
if Javascript is not None and display is not None:
    sync_js = r'''
(() => {
  const start = Date.now();
  const timeoutMs = 10000;
  const pollMs = 120;

  function findLatestPlot() {
    const plots = Array.from(document.querySelectorAll('.js-plotly-plot, .plotly-graph-div'));
    return plots.length ? plots[plots.length - 1] : null;
  }

  function bind() {
    const gd = findLatestPlot();
    if (!gd) {
      if (Date.now() - start < timeoutMs) setTimeout(bind, pollMs);
      return;
    }

    if (gd.__sceneCameraLinked) return;
    gd.__sceneCameraLinked = true;

    let syncing = false;

    const getSceneCam = (name) => {
      try {
        const sc = (gd._fullLayout && gd._fullLayout[name]) || (gd.layout && gd.layout[name]);
        if (!sc) return null;
        if (sc.camera) return sc.camera;
        if (sc._scene && typeof sc._scene.getCamera === 'function') return sc._scene.getCamera();
      } catch (e) {}
      return null;
    };

    const relay = (src, dst) => {
      const cam = getSceneCam(src);
      if (!cam) return;
      syncing = true;
      Plotly.relayout(gd, { [dst + '.camera']: cam })
        .catch(() => {})
        .finally(() => { syncing = false; });
    };

    gd.on('plotly_relayout', (ev) => {
      if (syncing || !ev) return;
      const keys = Object.keys(ev);
      const touchedScene = keys.some(k => k === 'scene.camera' || k.startsWith('scene.camera.'));
      const touchedScene2 = keys.some(k => k === 'scene2.camera' || k.startsWith('scene2.camera.'));
      if (touchedScene && !touchedScene2) relay('scene', 'scene2');
      else if (touchedScene2 && !touchedScene) relay('scene2', 'scene');
    });

    console.log('3D camera link active: scene <-> scene2');
  }

  bind();
})();
    '''
    display(Javascript(sync_js))
else:
    print("Javascript display is unavailable; camera interlock could not be attached.")


In [ ]:
# Random forest 3D+contour was merged into the cell above with LR.
# Cameras are interlocked there (scene <-> scene2).


In [ ]:
# Logistic regression PR AUC vs K: mean 95% PI + proper LOWESS fit (combined_ind style)
required = ["np", "pd", "go", "Path"]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError("Run setup/import cells first. Missing: " + ", ".join(missing))

# Ensure proper LOWESS implementation is available.
try:
    from statsmodels.nonparametric.smoothers_lowess import lowess
except Exception:
    import subprocess
    import sys
    import importlib

    print("Installing statsmodels for proper LOWESS...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "statsmodels"])
    importlib.invalidate_caches()
    from statsmodels.nonparametric.smoothers_lowess import lowess

lr_raw_path = K55_DIR / "lr_k55_truncation_out" / "truncation_logreg_raw.parquet"
if not lr_raw_path.exists():
    raise FileNotFoundError(f"Missing file: {lr_raw_path}")

metric = "test_PR_AUC"
df_lr = pd.read_parquet(lr_raw_path)
if metric not in df_lr.columns or "K" not in df_lr.columns:
    raise RuntimeError(f"Expected columns 'K' and '{metric}' in {lr_raw_path}")

plot_df = df_lr[["K", metric]].dropna().copy()
plot_df["K"] = plot_df["K"].astype(float)
plot_df["K_idx"] = plot_df["K"] - float(plot_df["K"].min())
plot_df = plot_df.sort_values("K_idx")

# Keep all points for mean/CI/point connections,
# but exclude the first index from LOWESS fitting only.
k_min = float(plot_df["K_idx"].min())
fit_df = plot_df[plot_df["K_idx"] > k_min].copy()
if fit_df.empty:
    raise RuntimeError("No data left for LOWESS after excluding the first index point.")

summ = (
    plot_df.groupby("K_idx", as_index=False)[metric]
    .agg(mean="mean", std="std", n="size")
    .sort_values("K_idx")
)
summ["std"] = summ["std"].fillna(0.0)
summ["se"] = summ["std"] / np.sqrt(summ["n"].clip(lower=1))

# t-based 95% PI when SciPy is available; normal approx fallback otherwise.
try:
    from scipy.stats import t as student_t
    tcrit = np.array([student_t.ppf(0.975, max(int(nn) - 1, 1)) for nn in summ["n"]], dtype=float)
except Exception:
    tcrit = np.full(len(summ), 1.96, dtype=float)

summ["ci95_lo"] = summ["mean"] - tcrit * summ["se"]
summ["ci95_hi"] = summ["mean"] + tcrit * summ["se"]

# Proper LOWESS on raw repeat-level points (same idea as CNN_truncation_light).
x_raw = fit_df["K_idx"].to_numpy(float)
y_raw = fit_df[metric].to_numpy(float)
sm = lowess(y_raw, x_raw, frac=0.22, it=0, return_sorted=True)
sm_df = pd.DataFrame(sm, columns=["K_idx", "smooth"]).groupby("K_idx", as_index=False)["smooth"].mean()

xd = np.linspace(float(fit_df["K_idx"].min()), float(plot_df["K_idx"].max()), 400)
yd = np.interp(xd, sm_df["K_idx"].to_numpy(float), sm_df["smooth"].to_numpy(float))

# Axis scaling based on K>1 only (do not let first point set vertical scale).
ref = summ[summ["K_idx"] > k_min].copy()
if ref.empty:
    ref = summ.copy()
ref_vals = np.concatenate([
    ref["ci95_lo"].to_numpy(float),
    ref["ci95_hi"].to_numpy(float),
    yd.astype(float),
])
ref_vals = ref_vals[np.isfinite(ref_vals)]
if ref_vals.size:
    y_lo_ref = float(np.min(ref_vals))
    y_hi_ref = float(np.max(ref_vals))
else:
    y_lo_ref, y_hi_ref = float(summ["mean"].min()), float(summ["mean"].max())
y_pad = max((y_hi_ref - y_lo_ref) * 0.08, 8e-4)
y_axis_range = [y_lo_ref - y_pad, y_hi_ref + y_pad]

# Blue palette aligned with combined_ind collage visuals.
COL_LOWESS = "#0b3c5d"
COL_MEAN = "#2c7fb8"
COL_MARKER = "#225d83"
COL_CI = "rgba(44,127,184,0.16)"

fig_lr_curve = go.Figure()

# 95% PI ribbon
fig_lr_curve.add_trace(
    go.Scatter(
        x=summ["K_idx"],
        y=summ["ci95_hi"],
        mode="lines",
        line=dict(width=0),
        hoverinfo="skip",
        showlegend=False,
    )
)
fig_lr_curve.add_trace(
    go.Scatter(
        x=summ["K_idx"],
        y=summ["ci95_lo"],
        mode="lines",
        line=dict(width=0),
        fill="tonexty",
        fillcolor=COL_CI,
        name="Vidurkio 95% PI",
        hovertemplate="Indeksas=%{x}<br>PI low=%{y:.4f}<extra></extra>",
    )
)

# Mean points/line
fig_lr_curve.add_trace(
    go.Scatter(
        x=summ["K_idx"],
        y=summ["mean"],
        mode="lines+markers",
        name="Vidurkis",
        line=dict(color=COL_MEAN, width=2.4),
        marker=dict(size=6, color=COL_MARKER, line=dict(color="white", width=0.7)),
        hovertemplate="Indeksas=%{x}<br>PR_AUC_mean=%{y:.4f}<extra></extra>",
    )
)

# Proper LOWESS fit
fig_lr_curve.add_trace(
    go.Scatter(
        x=xd,
        y=yd,
        mode="lines",
        name="LOWESS aproksimacija",
        line=dict(color=COL_LOWESS, width=4.3, shape="spline", smoothing=1.0),
        hovertemplate="Indeksas=%{x:.2f}<br>lowess=%{y:.4f}<extra></extra>",
    )
)

fig_lr_curve.update_layout(
    template="plotly_white",
    width=1900,
    height=850,
    font=dict(family="Avenir Next, Inter, Segoe UI, sans-serif", size=16),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=0.965,
        xanchor="right",
        x=0.99,
        bgcolor="rgba(255,255,255,0.80)",
        bordercolor="rgba(90,110,130,0.22)",
        borderwidth=1,
        traceorder="reversed",
    ),
    margin=dict(l=90, r=36, t=24, b=92),
)
x_ticks = np.unique(np.r_[np.arange(0.0, 55.0, 5.0), 54.0])
fig_lr_curve.update_xaxes(
    title_text="Aproksimuojančio polinomo koeficientų indeksas (K)",
    title_font=dict(size=26),
    tickfont=dict(size=22),
    tickmode="array",
    tickvals=x_ticks,
    ticktext=[str(int(v)) for v in x_ticks],
    range=[-0.2, 54.2],
    showgrid=True,
    gridcolor="rgba(90,110,130,0.16)",
    zeroline=False,
)
fig_lr_curve.update_yaxes(
    title_text="PR AUC",
    title_font=dict(size=26),
    tickfont=dict(size=22),
    showgrid=True,
    gridcolor="rgba(90,110,130,0.16)",
    zeroline=False,
    tickformat=".3f",
    range=y_axis_range,
)

_show_fig_hires(fig_lr_curve, filename="logistic_regression_pr_auc_vs_k", width=1900, height=850)



In [ ]:
# Random Forest PR AUC vs K: mean 95% PI + proper LOWESS fit (combined_ind style)
required = ["np", "pd", "go", "Path"]
missing = [k for k in required if k not in globals()]
if missing:
    raise RuntimeError("Run setup/import cells first. Missing: " + ", ".join(missing))

# Ensure proper LOWESS implementation is available.
try:
    from statsmodels.nonparametric.smoothers_lowess import lowess
except Exception:
    import subprocess
    import sys
    import importlib

    print("Installing statsmodels for proper LOWESS...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "statsmodels"])
    importlib.invalidate_caches()
    from statsmodels.nonparametric.smoothers_lowess import lowess

rf_raw_path = K55_DIR / "rf_k55_truncation_out" / "truncation_rf_raw.parquet"
if not rf_raw_path.exists():
    raise FileNotFoundError(f"Missing file: {rf_raw_path}")

metric = "test_PR_AUC"
df_rf = pd.read_parquet(rf_raw_path)
if metric not in df_rf.columns or "K" not in df_rf.columns:
    raise RuntimeError(f"Expected columns 'K' and '{metric}' in {rf_raw_path}")

plot_df = df_rf[["K", metric]].dropna().copy()
plot_df["K"] = plot_df["K"].astype(float)
plot_df["K_idx"] = plot_df["K"] - float(plot_df["K"].min())
plot_df = plot_df.sort_values("K_idx")

# Keep all points for mean/CI/point connections,
# but exclude the first index from LOWESS fitting only.
k_min = float(plot_df["K_idx"].min())
fit_df = plot_df[plot_df["K_idx"] > k_min].copy()
if fit_df.empty:
    raise RuntimeError("No data left for LOWESS after excluding the first index point.")

summ = (
    plot_df.groupby("K_idx", as_index=False)[metric]
    .agg(mean="mean", std="std", n="size")
    .sort_values("K_idx")
)
summ["std"] = summ["std"].fillna(0.0)
summ["se"] = summ["std"] / np.sqrt(summ["n"].clip(lower=1))

# t-based 95% PI when SciPy is available; normal approx fallback otherwise.
try:
    from scipy.stats import t as student_t
    tcrit = np.array([student_t.ppf(0.975, max(int(nn) - 1, 1)) for nn in summ["n"]], dtype=float)
except Exception:
    tcrit = np.full(len(summ), 1.96, dtype=float)

summ["ci95_lo"] = summ["mean"] - tcrit * summ["se"]
summ["ci95_hi"] = summ["mean"] + tcrit * summ["se"]

# Proper LOWESS on raw repeat-level points.
x_raw = fit_df["K_idx"].to_numpy(float)
y_raw = fit_df[metric].to_numpy(float)
sm = lowess(y_raw, x_raw, frac=0.22, it=0, return_sorted=True)
sm_df = pd.DataFrame(sm, columns=["K_idx", "smooth"]).groupby("K_idx", as_index=False)["smooth"].mean()

xd = np.linspace(float(fit_df["K_idx"].min()), float(plot_df["K_idx"].max()), 400)
yd = np.interp(xd, sm_df["K_idx"].to_numpy(float), sm_df["smooth"].to_numpy(float))

# Axis scaling based on K>1 only (do not let first point set vertical scale).
ref = summ[summ["K_idx"] > k_min].copy()
if ref.empty:
    ref = summ.copy()
ref_vals = np.concatenate([
    ref["ci95_lo"].to_numpy(float),
    ref["ci95_hi"].to_numpy(float),
    yd.astype(float),
])
ref_vals = ref_vals[np.isfinite(ref_vals)]
if ref_vals.size:
    y_lo_ref = float(np.min(ref_vals))
    y_hi_ref = float(np.max(ref_vals))
else:
    y_lo_ref, y_hi_ref = float(summ["mean"].min()), float(summ["mean"].max())
y_pad = max((y_hi_ref - y_lo_ref) * 0.08, 8e-4)
y_axis_range = [y_lo_ref - y_pad, y_hi_ref + y_pad]

# Blue palette aligned with combined_ind collage visuals.
COL_LOWESS = "#0b3c5d"
COL_MEAN = "#2c7fb8"
COL_MARKER = "#225d83"
COL_CI = "rgba(44,127,184,0.16)"

fig_rf_curve = go.Figure()

# 95% PI ribbon
fig_rf_curve.add_trace(
    go.Scatter(
        x=summ["K_idx"],
        y=summ["ci95_hi"],
        mode="lines",
        line=dict(width=0),
        hoverinfo="skip",
        showlegend=False,
    )
)
fig_rf_curve.add_trace(
    go.Scatter(
        x=summ["K_idx"],
        y=summ["ci95_lo"],
        mode="lines",
        line=dict(width=0),
        fill="tonexty",
        fillcolor=COL_CI,
        name="Vidurkio 95% PI",
        hovertemplate="Indeksas=%{x}<br>PI low=%{y:.4f}<extra></extra>",
    )
)

# Mean points/line
fig_rf_curve.add_trace(
    go.Scatter(
        x=summ["K_idx"],
        y=summ["mean"],
        mode="lines+markers",
        name="Vidurkis",
        line=dict(color=COL_MEAN, width=2.4),
        marker=dict(size=6, color=COL_MARKER, line=dict(color="white", width=0.7)),
        hovertemplate="Indeksas=%{x}<br>PR_AUC_mean=%{y:.4f}<extra></extra>",
    )
)

# Proper LOWESS fit
fig_rf_curve.add_trace(
    go.Scatter(
        x=xd,
        y=yd,
        mode="lines",
        name="LOWESS aproksimacija",
        line=dict(color=COL_LOWESS, width=4.3, shape="spline", smoothing=1.0),
        hovertemplate="Indeksas=%{x:.2f}<br>lowess=%{y:.4f}<extra></extra>",
    )
)

fig_rf_curve.update_layout(
    template="plotly_white",
    width=1900,
    height=850,
    font=dict(family="Avenir Next, Inter, Segoe UI, sans-serif", size=16),
    legend=dict(
        orientation="h",
        yanchor="top",
        y=0.965,
        xanchor="right",
        x=0.99,
        bgcolor="rgba(255,255,255,0.80)",
        bordercolor="rgba(90,110,130,0.22)",
        borderwidth=1,
        traceorder="reversed",
    ),
    margin=dict(l=90, r=36, t=24, b=92),
)
x_ticks = np.unique(np.r_[np.arange(0.0, 55.0, 5.0), 54.0])
fig_rf_curve.update_xaxes(
    title_text="Aproksimuojančio polinomo koeficientų indeksas (K)",
    title_font=dict(size=26),
    tickfont=dict(size=22),
    tickmode="array",
    tickvals=x_ticks,
    ticktext=[str(int(v)) for v in x_ticks],
    range=[-0.2, 54.2],
    showgrid=True,
    gridcolor="rgba(90,110,130,0.16)",
    zeroline=False,
)
fig_rf_curve.update_yaxes(
    title_text="PR AUC",
    title_font=dict(size=26),
    tickfont=dict(size=22),
    showgrid=True,
    gridcolor="rgba(90,110,130,0.16)",
    zeroline=False,
    tickformat=".3f",
    range=y_axis_range,
)

_show_fig_hires(fig_rf_curve, filename="random_forest_pr_auc_vs_k", width=1900, height=850)
